In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# 注意这里checkpointer是langgraph的
from langgraph.checkpoint.memory import InMemorySaver


# 模型实例
llm = ChatOpenAI(model="qwen3.5-flash", temperature=0.7, max_tokens=10000)

# 创建agent
agent = create_agent(
    model=llm,        # 模型可以是 "provider:model" 字符串标识，也可模型实例
    tools=[],
    checkpointer=InMemorySaver() # 持久化：按 thread_id 持久化 State
)

# 运行时配置
# configurable.thread_id 是给 checkpointer 用的会话标识 
config = {'configurable': {'thread_id': '10010'}}

result = agent.invoke(
    {'messages': [{'role': 'user', 'content': '你好, 我是limbo 你是？'}]},
    config=config
)

result

{'messages': [HumanMessage(content='你好, 我是limbo 你是？', additional_kwargs={}, response_metadata={}, id='3bc81f40-8748-4aaa-bd97-3b90eb483c17'),
  AIMessage(content='你好 Limbo！很高兴认识你～\n\n我是 **Qwen3.5**，是阿里巴巴通义实验室研发的超大规模语言模型。你也可以叫我 Qwen 或者通义千问。\n\n无论是聊天解闷、创作内容、解答疑惑，还是处理任务（比如写代码、分析文档等），我都很乐意帮忙。今天有什么想聊的或者需要协助的吗？😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1021, 'prompt_tokens': 19, 'total_tokens': 1040, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 936, 'rejected_prediction_tokens': None, 'text_tokens': 1021}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 19}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-flash', 'system_fingerprint': None, 'id': 'chatcmpl-3c3c91ff-3746-90a0-bb9d-9dfc44070571', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eb09c-6741-7d81-86d8-203476d61e6d-0', tool_calls=[], invalid_tool_calls=[], usage_me

In [ ]:
# 同一个 thread_id 已经持久化消息了，下轮对话不需要手动添加历史消息
agent.invoke({"messages": [{"role": "user", "content": "我是谁？"}]}, config=config)

{'messages': [HumanMessage(content='你好, 我是limbo 你是？', additional_kwargs={}, response_metadata={}, id='3bc81f40-8748-4aaa-bd97-3b90eb483c17'),
  AIMessage(content='你好 Limbo！很高兴认识你～\n\n我是 **Qwen3.5**，是阿里巴巴通义实验室研发的超大规模语言模型。你也可以叫我 Qwen 或者通义千问。\n\n无论是聊天解闷、创作内容、解答疑惑，还是处理任务（比如写代码、分析文档等），我都很乐意帮忙。今天有什么想聊的或者需要协助的吗？😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1021, 'prompt_tokens': 19, 'total_tokens': 1040, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 936, 'rejected_prediction_tokens': None, 'text_tokens': 1021}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 19}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-flash', 'system_fingerprint': None, 'id': 'chatcmpl-3c3c91ff-3746-90a0-bb9d-9dfc44070571', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eb09c-6741-7d81-86d8-203476d61e6d-0', tool_calls=[], invalid_tool_calls=[], usage_me

In [ ]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt
from langchain.tools import tool, ToolRuntime


@dataclass
class Context:
    user_id: str
    vip_level: int
    user_name: str
    

@tool
def get_my_orders(runtime: ToolRuntime[Context]) -> str:
    """
    获取用户的订单信息
    """
    uid = runtime.context.user_id
    print(f"工具调用：get_my_orders, ctx: {runtime.context}")
    if uid == '1001':
        return f"用户 {runtime.context.user_name} 的订单支付了100美金"
    else:
        return "订单不存在"


@dynamic_prompt
def personalized(request) -> str:
    ctx = request.runtime.context
    base = f"你是客服助理，称呼用户为{ctx.user_name}。"
    if ctx.vip_level >= 3:
        base += "该用户是 VIP， 称呼时候需要加上尊贵的VIP用户"
    return base


# 创建agent
agent = create_agent(
    model=llm,        # 模型可以是 "provider:model" 字符串标识，也可模型实例
    tools=[get_my_orders],
    context_schema=Context,
    middleware=[personalized],
    checkpointer=InMemorySaver() # 持久化：按 thread_id 持久化 State
)

# 运行时配置
# configurable.thread_id 是给 checkpointer 用的会话标识 
config = {'configurable': {'thread_id': '10010'}}


# thread_id 限定对话范围（消息历史、检查点）
# 而 context 承载工具和中间件在调用时读取的每次运行数据。二者通常会一并传递
result = agent.invoke(
    {'messages': [{'role': 'user', 'content': '你好, 看下我的订单'}]},
    config=config,
    context=Context(user_id='1001', vip_level=5, user_name='方先生')
)

result

工具调用：get_my_orders, ctx: Context(user_id='1001', vip_level=5, user_name='方先生')


{'messages': [HumanMessage(content='你好, 看下我的订单', additional_kwargs={}, response_metadata={}, id='b38a05ca-4ec0-4515-b174-2ef8009d06ab'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 283, 'total_tokens': 346, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 44, 'rejected_prediction_tokens': None, 'text_tokens': 63}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 283}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-flash', 'system_fingerprint': None, 'id': 'chatcmpl-2f848235-89b3-90fd-b3e6-dfc2225f97d7', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eb0d4-076c-7f31-a357-24cbc459f211-0', tool_calls=[{'name': 'get_my_orders', 'args': {}, 'id': 'call_73d41948f26d4665af0f9a96', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 283, 'output_tokens': 63, 